# Playback review

Two questions about one playback run:

1. **What did the Cartesian loop do?** Issued command → controller setpoint → measured pose,
   plus twist, acceleration and external wrench.
2. **Does it match the demonstration it replays?**

Three sources, resolved automatically from the run log:

| source | what it holds | rate |
|---|---|---|
| `logs/run_*.hdf5` | what PlaybackModule **issued**, and the tick → source-time mapping | orchestrator tick (~20 Hz) |
| `teleop-simulator/logs/NNN/` | what the arm **did**: pose, setpoint, wrench, joints | 500–1000 Hz |
| store `NNN/episode.hdf5` | the **demonstration** being replayed | 30 Hz |

The avatar folder matters because the run log only records measured *position* — no
orientation, no wrench. The simulator writes all of it per run anyway.


In [ ]:
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from scipy.spatial.transform import Rotation

ORCH_LOGS = Path("../logs")
SIM_LOGS  = Path("../../teleop-simulator/logs")

ARMS = ("arm_left", "arm_right")
ENGAGED = 4
ARM = "arm_left"          # section 1 renders one arm at a time
plt.rcParams["figure.figsize"] = (13, 7)

for f in sorted(ORCH_LOGS.glob("run_*.hdf5"))[-8:]:
    print(f.stem, f"{f.stat().st_size / 1e6:6.1f} MB")

In [ ]:
RUN = "run_20260914_144308"

# Both are resolved below; set either to override.
SOURCE_EPISODE = None     # e.g. "C:/Users/ceti/.../avatar/014/episode.hdf5"
AVATAR_FOLDER  = None     # e.g. SIM_LOGS / "033"

## Load

In [ ]:
def _pose(flat16):
    """Column-major flat 4x4 -> (pos [N,3], Rotation[N])."""
    m = np.asarray(flat16).reshape(-1, 4, 4).transpose(0, 2, 1)
    return m[:, :3, 3].copy(), Rotation.from_matrix(m[:, :3, :3])


def _wall_range(csv_path):
    """(first, last) wall_clock_ns without parsing the whole file."""
    with open(csv_path) as f:
        header = f.readline().strip().split(";")
        col = header.index("wall_clock_ns")
        first = f.readline().split(";")[col]
        f.seek(max(0, csv_path.stat().st_size - 4096))
        tail = [ln for ln in f.read().splitlines() if ln.count(";") == len(header) - 1]
    return int(float(first)), int(float(tail[-1].split(";")[col]))


def load_run(name):
    """Issued commands + the wall-clock -> source-time mapping."""
    out = {"arms": {}}
    with h5py.File(ORCH_LOGS / f"{name}.hdf5", "r") as f:
        out["attrs"] = dict(f.attrs)
        mod = "playback" if "modules/playback" in f else "policy"
        out["module"] = mod
        n = len(f[f"modules/{mod}/arm_left/ee_pose"])
        out["t_wall"] = f["frames/timestamp_ns"][:n].astype(np.int64)
        out["src_t"] = (f[f"modules/{mod}/extras/playback_t_s"][:n]
                        if f"modules/{mod}/extras/playback_t_s" in f else None)
        for arm in ARMS:
            ee = f[f"modules/{mod}/{arm}/ee_pose"][:]
            out["arms"][arm] = {
                "pos": ee[:, :3],
                "rot": Rotation.from_quat(ee[:, [4, 5, 6, 3]]),   # stored w,x,y,z
                "gripper": f[f"modules/{mod}/{arm}/gripper"][:],
            }
    return out


def load_avatar(folder):
    """What the arms actually did, from the simulator's own per-run CSVs."""
    out = {}
    for arm in ARMS:
        df = pd.read_csv(folder / f"{arm}.csv", sep=";").dropna(subset=["state"])
        cols = lambda p, n: df[[f"{p}_{i}" for i in range(n)]].to_numpy()
        pos, rot = _pose(cols("O_T_EE_world", 16))
        spos, srot = _pose(cols("O_T_EE_cmd_world", 16))
        out[arm] = {
            "t_wall": df["wall_clock_ns"].to_numpy().astype(np.int64),
            "pos": pos, "rot": rot,              # measured
            "set_pos": spos, "set_rot": srot,    # motion-generator setpoint
            "F_ext": cols("F_ext", 6),
            "q": cols("q", 7), "dq": cols("dq", 7), "tau_ext": cols("tau_ext", 7),
            "gripper_width": df["gripper_width"].to_numpy(),
            "gripper_cmd": df["gripper_cmd"].to_numpy(),
            "state": df["state"].to_numpy().astype(int),
        }
        out[arm]["t"] = (out[arm]["t_wall"] - out[arm]["t_wall"][0]) * 1e-9
    return out


def load_source(path):
    """The demonstration, clipped to the window PlaybackModule replays (both arms
    ENGAGED) so its t=0 is the same instant as playback_t_s=0."""
    out = {}
    with h5py.File(path, "r") as f:
        ts = f["observations/timestamp_ns"][:].astype(np.int64)
        eng = np.ones(len(ts), bool)
        for arm in ARMS:
            key = f"observations/{arm}/state"
            if key in f:
                eng &= f[key][:].astype(int) == ENGAGED
        idx = np.flatnonzero(eng)
        sl = slice(int(idx[0]), int(idx[-1]) + 1)
        t = (ts[sl] - ts[sl][0]) * 1e-9
        for arm in ARMS:
            g, a = f[f"observations/{arm}"], f[f"actions/{arm}"]
            pos, rot = _pose(g["O_T_EE_world"][sl])
            cpos, crot = _pose(a["O_T_EE_cmd_world"][sl])
            out[arm] = {"t": t, "pos": pos, "rot": rot, "cmd_pos": cpos, "cmd_rot": crot,
                        "F_ext": g["F_ext"][sl], "gripper_width": g["gripper_width"][sl],
                        "gripper_cmd": a["gripper_cmd"][sl]}
    return out


run = load_run(RUN)
print(run["attrs"])

src_path = SOURCE_EPISODE or run["attrs"].get("playback_episode")
source = load_source(src_path) if src_path else None
print("source :", src_path)

folder = AVATAR_FOLDER
if folder is None:
    t0, t1 = run["t_wall"][0], run["t_wall"][-1]
    for d in sorted(SIM_LOGS.glob("[0-9][0-9][0-9]")):
        if not (d / "arm_left.csv").exists():
            continue
        a, b = _wall_range(d / "arm_left.csv")
        if a <= t1 and b >= t0:          # overlaps this run in wall-clock time
            folder = d
avatar = load_avatar(folder) if folder else None
print("avatar :", folder)

# Wall clock -> source-episode time, taken straight from the run log.
to_src_t = (lambda w: np.interp(w.astype(float), run["t_wall"].astype(float), run["src_t"])) \
    if run["src_t"] is not None else None

## 1 — Cartesian behaviour of this run

Three traces everywhere it matters:

- **issued** — what PlaybackModule put on the wire, at the orchestrator tick rate.
- **setpoint** — `O_T_EE_cmd_world`, the motion generator's interpolated target after
  rate and acceleration limiting. Not the same thing as the issued command.
- **measured** — `O_T_EE_world`.

issued → setpoint is command shaping; setpoint → measured is controller tracking. The
two are worth keeping apart: the end-to-end error is dominated by the first.

In [ ]:
R = run["arms"][ARM]

# Clip the avatar telemetry to this run's ENGAGED window. Outside it the controller
# setpoint is a leftover (the arm is homing or parked), which plots as a metre-scale
# step and hides everything real.
_A = avatar[ARM]
win = ((_A["state"] == ENGAGED)
       & (_A["t_wall"] >= run["t_wall"][0]) & (_A["t_wall"] <= run["t_wall"][-1]))
A = {k: v[win] for k, v in _A.items()}
t0 = A["t_wall"][0]
A["t"] = (A["t_wall"] - t0) * 1e-9
t_run = (run["t_wall"] - t0) * 1e-9
print(f"{ARM}: {win.sum()} samples, {A['t'][-1]:.1f} s engaged, "
      f"{1 / np.median(np.diff(A['t'])):.0f} Hz")


def grid(axs):
    for ax in np.atleast_1d(axs).ravel():
        ax.grid(True, alpha=0.3)
    if np.atleast_1d(axs)[0].get_legend_handles_labels()[0]:
        np.atleast_1d(axs)[0].legend(ncol=4, fontsize=8)
    np.atleast_1d(axs)[-1].set_xlabel("t [s]")


def deriv(t, x, smooth=25):
    """Finite difference with a moving average; these are 500 Hz signals."""
    d = np.gradient(x, t, axis=0)
    if smooth > 1:
        k = np.ones(smooth) / smooth
        d = np.apply_along_axis(lambda c: np.convolve(c, k, mode="same"), 0, d)
    return d


def ang_vel(t, rot, smooth=25):
    """Angular velocity from consecutive rotations (rotvec, stable near identity)."""
    dR = (rot[1:] * rot[:-1].inv()).as_rotvec() / np.diff(t)[:, None]
    dR = np.vstack([dR[:1], dR])
    k = np.ones(smooth) / smooth
    return np.apply_along_axis(lambda c: np.convolve(c, k, mode="same"), 0, dR)


fig, axs = plt.subplots(3, 1, sharex=True, figsize=(13, 6))
for i, lbl in enumerate("xyz"):
    axs[i].plot(t_run, R["pos"][:, i], lw=1.2, color="tab:blue", label="issued")
    axs[i].plot(A["t"], A["set_pos"][:, i], lw=0.9, color="tab:orange", label="setpoint")
    axs[i].plot(A["t"], A["pos"][:, i], lw=0.9, ls="--", color="tab:red", label="measured")
    axs[i].set_ylabel(f"{lbl} [m]")
grid(axs)
plt.suptitle(f"{RUN} / {ARM} — EE translation (world frame)")
plt.tight_layout()

In [ ]:
fig, axs = plt.subplots(4, 1, sharex=True, figsize=(13, 7))
qr, qs, qm = R["rot"].as_quat(), A["set_rot"].as_quat(), A["rot"].as_quat()
for i, lbl in enumerate(["x", "y", "z", "w"]):
    axs[i].plot(t_run, qr[:, i], lw=1.2, color="tab:blue", label="issued")
    axs[i].plot(A["t"], qs[:, i], lw=0.9, color="tab:orange", label="setpoint")
    axs[i].plot(A["t"], qm[:, i], lw=0.9, ls="--", color="tab:red", label="measured")
    axs[i].set_ylabel(f"q{lbl}")
grid(axs)
plt.suptitle(f"{RUN} / {ARM} — EE orientation (quaternion, world frame)")
plt.tight_layout()

In [ ]:
# Tracking error, split into the two stages.
issued_pos = np.stack([np.interp(A["t"], t_run, R["pos"][:, i]) for i in range(3)], 1)
issued_rot = Rotation.from_quat(np.stack([np.interp(A["t"], t_run, qr[:, i]) for i in range(4)], 1)
                                / np.linalg.norm([np.interp(A["t"], t_run, qr[:, i]) for i in range(4)], axis=0)[:, None])

fig, axs = plt.subplots(2, 1, sharex=True, figsize=(13, 5))
axs[0].plot(A["t"], np.linalg.norm(issued_pos - A["pos"], axis=1) * 1e3, lw=0.9, label="issued -> measured")
axs[0].plot(A["t"], np.linalg.norm(A["set_pos"] - A["pos"], axis=1) * 1e3, lw=0.9, label="setpoint -> measured")
axs[0].set_ylabel("|dp| [mm]")
axs[1].plot(A["t"], np.degrees((issued_rot * A["rot"].inv()).magnitude()), lw=0.9, label="issued -> measured")
axs[1].plot(A["t"], np.degrees((A["set_rot"] * A["rot"].inv()).magnitude()), lw=0.9, label="setpoint -> measured")
axs[1].set_ylabel("angle [deg]")
grid(axs)
plt.suptitle(f"{RUN} / {ARM} — tracking error")
plt.tight_layout()

In [ ]:
# Twist. Differentiated from the logged poses, not from a Jacobian. The sawtooth on
# the setpoint traces is real: commands arrive at the orchestrator tick rate and the
# motion generator replans a linear segment on each one, so its velocity steps.
v_m, v_s = deriv(A["t"], A["pos"]), deriv(A["t"], A["set_pos"])
w_m, w_s = ang_vel(A["t"], A["rot"]), ang_vel(A["t"], A["set_rot"])

fig, axs = plt.subplots(2, 1, sharex=True, figsize=(13, 6))
for i, lbl in enumerate("xyz"):
    axs[0].plot(A["t"], v_s[:, i], lw=0.6, alpha=0.4, label=f"setpoint {lbl}")
    axs[0].plot(A["t"], v_m[:, i], lw=1.0, label=f"measured {lbl}")
    axs[1].plot(A["t"], w_s[:, i], lw=0.6, alpha=0.4, label=f"setpoint {lbl}")
    axs[1].plot(A["t"], w_m[:, i], lw=1.0, label=f"measured {lbl}")
axs[0].set_ylabel("v [m/s]")
axs[1].set_ylabel("w [rad/s]")
grid(axs)
axs[0].legend(ncol=6, fontsize=7)
plt.suptitle(f"{RUN} / {ARM} — Cartesian twist")
plt.tight_layout()

In [ ]:
# Acceleration. Double-differentiated, so read it for shape and timing, not magnitude --
# and expect the setpoint trace to show the tick rate the commands arrive at.
a_m, a_s = deriv(A["t"], v_m), deriv(A["t"], v_s)

fig, axs = plt.subplots(2, 1, sharex=True, figsize=(13, 6))
for i, lbl in enumerate("xyz"):
    axs[0].plot(A["t"], a_s[:, i], lw=0.8, label=lbl)
    axs[1].plot(A["t"], a_m[:, i], lw=0.8, label=lbl)
axs[0].set_ylabel("setpoint\na [m/s^2]")
axs[1].set_ylabel("measured\na [m/s^2]")
grid(axs)
plt.suptitle(f"{RUN} / {ARM} — Cartesian acceleration")
plt.tight_layout()

In [ ]:
fig, axs = plt.subplots(3, 1, sharex=True, figsize=(13, 7))
for i, lbl in enumerate("xyz"):
    axs[0].plot(A["t"], A["F_ext"][:, i], lw=0.8, label=f"F{lbl}")
    axs[1].plot(A["t"], A["F_ext"][:, 3 + i], lw=0.8, label=f"T{lbl}")
axs[0].set_ylabel("force [N]")
axs[1].set_ylabel("torque [Nm]")
axs[2].plot(A["t"], A["gripper_width"], lw=1.0, label="width")
axs[2].plot(A["t"], A["gripper_cmd"], lw=0.9, ls="--", label="cmd")
axs[2].set_ylabel("gripper [m]")
for ax in axs:
    ax.legend(ncol=3, fontsize=8)
grid(axs)
plt.suptitle(f"{RUN} / {ARM} — external wrench and gripper")
plt.tight_layout()

## 2 — Against the demonstration

Both runs are put on the demonstration's own clock: the run log's `playback_t_s` says which
instant of the source each tick was issuing, which maps the avatar's wall clock onto source
time. During the start ramp `playback_t_s` is pinned at 0, so that stretch collapses to a
point — ignore the first sample.

In [ ]:
S = source[ARM]
# A is already clipped to the run's ENGAGED window; drop the start ramp too, where
# playback_t_s is pinned at 0 and the mapping is not invertible.
t_src_of_avatar = to_src_t(A["t_wall"])
m = t_src_of_avatar > 0

# Demonstration resampled onto the same instants, for a like-for-like difference.
src_pos = np.stack([np.interp(t_src_of_avatar[m], S["t"], S["pos"][:, i]) for i in range(3)], 1)
sq = S["rot"].as_quat()
src_rot = Rotation.from_quat(np.stack([np.interp(t_src_of_avatar[m], S["t"], sq[:, i]) for i in range(4)], 1))

fig, axs = plt.subplots(3, 1, sharex=True, figsize=(13, 6))
for i, lbl in enumerate("xyz"):
    axs[i].plot(S["t"], S["pos"][:, i], lw=1.0, color="tab:grey", label="demo measured")
    axs[i].plot(t_src_of_avatar[m], A["pos"][m, i], lw=0.9, color="tab:red", label="playback measured")
    axs[i].plot(S["t"], S["cmd_pos"][:, i], lw=0.7, ls=":", color="tab:blue", label="demo command")
    axs[i].set_ylabel(f"{lbl} [m]")
grid(axs)
plt.suptitle(f"{ARM} — playback vs demonstration, on the demonstration's clock")
plt.tight_layout()

In [ ]:
fig, axs = plt.subplots(3, 1, sharex=True, figsize=(13, 6))
axs[0].plot(t_src_of_avatar[m], np.linalg.norm(A["pos"][m] - src_pos, axis=1) * 1e3, lw=0.9)
axs[0].set_ylabel("|dp| [mm]")
axs[1].plot(t_src_of_avatar[m], np.degrees((Rotation.from_quat(A["rot"].as_quat()[m]) * src_rot.inv()).magnitude()), lw=0.9)
axs[1].set_ylabel("angle [deg]")
axs[2].plot(t_src_of_avatar[m], A["gripper_width"][m], lw=1.0, color="tab:red", label="playback")
axs[2].plot(S["t"], S["gripper_width"], lw=0.9, color="tab:grey", label="demo")
axs[2].plot(S["t"], S["gripper_cmd"], lw=0.7, ls=":", color="tab:blue", label="demo cmd")
axs[2].set_ylabel("gripper [m]")
grid(axs)
plt.suptitle(f"{ARM} — divergence from the demonstration")
plt.tight_layout()

d = np.linalg.norm(A["pos"][m] - src_pos, axis=1) * 1e3
print(f"position divergence: median {np.median(d):6.1f} mm   p95 {np.percentile(d, 95):6.1f} mm   max {d.max():6.1f} mm")

In [ ]:
# Contact: if playback reproduces the demonstration, it should meet the same forces at
# roughly the same moments. Divergence here is the one that matters for the task.
fig, axs = plt.subplots(2, 1, sharex=True, figsize=(13, 5))
axs[0].plot(t_src_of_avatar[m], np.linalg.norm(A["F_ext"][m, :3], axis=1), lw=0.8, color="tab:red", label="playback")
axs[0].plot(S["t"], np.linalg.norm(S["F_ext"][:, :3], axis=1), lw=0.8, color="tab:grey", label="demo")
axs[0].set_ylabel("|F| [N]")
axs[1].plot(t_src_of_avatar[m], A["F_ext"][m, 2], lw=0.8, color="tab:red", label="playback")
axs[1].plot(S["t"], S["F_ext"][:, 2], lw=0.8, color="tab:grey", label="demo")
axs[1].set_ylabel("Fz [N]")
grid(axs)
plt.suptitle(f"{ARM} — external force, playback vs demonstration")
plt.tight_layout()